
# 02. Data Splitting

## Objective

Generate reproducible training and validation splits for EC and SA datasets.

## Configuration

- Random seeds: 42, 43, 44, 45, 46
- Validation ratio: 10%
- Sequence-disjoint splitting
- Independent test datasets preserved

## Workflow

1. Load EC and SA training datasets.
2. Generate validation splits for each random seed.
3. Verify sequence independence.
4. Save split indices for reuse across all models.

## Principles

- Preserve the original Train/Test datasets.
- Use identical splits for Traditional ML and PepBERT.
- Do not use test datasets for model selection.


In [2]:

import importlib
import pandas as pd
import numpy as np

from configs.common import DATA_DIR, RANDOM_SEEDS, VALID_SIZE
import src.dataset.splitting as splitting

# Reload updated module
importlib.reload(splitting)

# Output directory
split_dir = DATA_DIR / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

summary = []

for target in ["EC", "SA"]:

    # Load training dataset
    df = pd.read_csv(
        DATA_DIR / f"{target}_Train.csv"
    )

    # Generate validation splits
    splits = splitting.create_validation_splits(
        df,
        seeds=RANDOM_SEEDS,
        valid_size=VALID_SIZE
    )

    for seed, indices in splits.items():

        train_idx = indices["train"]
        valid_idx = indices["valid"]

        # Store row positions and subsets
        split_df = pd.DataFrame({
            "row_index": np.arange(len(df)),
            "subset": "train"
        })

        split_df.loc[valid_idx, "subset"] = "valid"

        # Save split indices
        output_path = (
            split_dir / f"{target}_seed{seed}.csv"
        )

        split_df.to_csv(output_path, index=False)

        summary.append({
            "Dataset": target,
            "Seed": seed,
            "Train": len(train_idx),
            "Validation": len(valid_idx),
            "Overlap": len(
                set(df.iloc[train_idx]["sequence"])
                & set(df.iloc[valid_idx]["sequence"])
            )
        })

# Display summary
summary_df = pd.DataFrame(summary)

display(summary_df)

# Verify all splits
assert (summary_df["Overlap"] == 0).all()

print("\\n[OK] Validation splits saved.")


,Dataset,Seed,Train,Validation,Overlap
0,EC,42,6824,759,0
1,EC,43,6824,759,0
2,EC,44,6824,759,0
3,EC,45,6824,759,0
4,EC,46,6824,759,0
5,SA,42,5932,660,0
6,SA,43,5932,660,0
7,SA,44,5932,660,0
8,SA,45,5932,660,0
9,SA,46,5932,660,0


\n[OK] Validation splits saved.
